In [ ]:
def check_proof_with_assumptions(proof,debug = False):
    proof = proof.splitlines()
    thesis_conclusion = parse_infix(proof[0][12:])
    base_context = []
    if proof[1][0] == "a":
        if proof[1][-1] == ",":
            base_context.append(parse_infix(proof[1][15:-1]))
            j = 2
            while proof[j][0] != "1":
                if proof[j][-1] == ",":
                    base_context.append(parse_infix(proof[j][:-1]))
                else:
                    base_context.append(parse_infix(proof[j]))
                j += 1
        else:
            base_context.append(parse_infix(proof[1][15:]))
    proof = "\n".join(proof[1+len(base_context):])
    parsedProof = parseProof(proof,base_context)
    if debug == True:
        print(parsedProof)

    if parsedProof[len(parsedProof)].LP != LittleProblem(Context(base_context,[]) , thesis_conclusion):
        raise TypeError("Bad proof")
    return parsedProof




In [ ]:
with open("proofs_normalised.txt", "w", encoding="utf-8") as f:
    for i in proofs_table:
        print(i)
        f.write(i + "\n\n")

In [ ]:
with open("proofs_normalised.txt", "r", encoding="utf-8") as f:
    text = f.read()
text = [p.strip() for p in re.split(r'\n\s*\n', text.strip()) if p.strip()]

In [ ]:
def almost_equal(x:LittleProblem,y:LittleProblem):
    return basing_context_of_LP(x) == basing_context_of_LP(y)

def extract_proof(proof:str,LP:LittleProblem,BaseContext = []):
    '''
    BaseContext to kontekst bazowy w którym dzieje się proof
    '''
    parsedProof = parseProof(proof,base_context=BaseContext)
    goal_idx = -1
    for i in parsedProof.keys():
        if almost_equal(parsedProof[i].LP,LP):
            goal_idx = i
            break
    if goal_idx == -1:
        raise TypeError("Cannot extract")
    important_lines = [goal_idx]
    def next_elems():

        ans = []
        for i in parsedProof.keys():
            if not i in important_lines:
                for j in  important_lines:
                    if i in parsedProof[j].args_keys:
                        ans.append(i)
                        break

        return ans
    next_elems_values = next_elems()
    while next_elems_values != []:
        important_lines += next_elems_values
        next_elems_values = next_elems()
    important_lines.sort()
    ans = ""
    proof = proof.splitlines()
    for i in proof:
        idx = int(i.split(".")[0])
        if idx in important_lines:
            ans += i
            ans += "\n"
    ans = ans[:-1]
    return ans
    def replace_indexed_numbers(ls: List[str], li: List[int]) -> List[str]:
        mapping = {int(li[i]): str(i+1) for i in range(len(li))}
        if not mapping:
            return ls[:]

        alts = "|".join(
            re.escape(str(n)) for n in sorted(mapping.keys(), key=lambda x: (-len(str(x)), x))
        )

        pattern = re.compile(
            rf'(?:(, |–)({alts})(?!\d))|(?:(?<!\d)({alts})(?!\d)(?=\.))'
        )

        def repl(m: re.Match) -> str:
            if m.group(2) is not None:  # gałąź z prefiksem
                prefix = m.group(1)
                num = m.group(2)
                return prefix + mapping[int(num)]
            else:  # gałąź z kropką po liczbie
                num = m.group(3)
                return mapping[int(num)]

        ans_lines = [pattern.sub(repl, s) for s in ls]
        ans = "\n".join(ans_lines)
        return ans

    return replace_indexed_numbers(ans.splitlines(),important_lines)

def give_all_extraction_text(proof:str,BaseContext = []):
    parsedProof = parseProof(proof,base_context = BaseContext)
    ANS = ""
    for i in parsedProof.keys():
        goal = parsedProof[i].LP
        temporary_context = deepcopy(BaseContext)
        for j in goal.assumptions.base_context:
            if not j in BaseContext:
                temporary_context.append(j)
        for j in goal.assumptions.additional_context:
            if not j in BaseContext:
                temporary_context.append(j)
        goal.assumptions.base_context = temporary_context#?
        goal.assumptions.additional_context = []#?
        extracted = extract_proof(proof,goal,temporary_context)
        parseProof(extracted,temporary_context)
        #extracted = simpliffy_with_context(extracted,temporary_context)
        parseProof(extracted,base_context = temporary_context)
        ans = "Prove that: " + to_infix(goal.conclusion)
        if temporary_context != []:
            ans += "\nassuming that: "
            for j in temporary_context:
                ans += to_infix(j)
                ans += ",\n"
            ans = ans[:-2]
        ans += ("\n"+extracted)
        ANS += ans
        ANS += "\n\n"
    return ANS


def _extract_and_check_one(proof_with_goal):
    proof = "\n".join(proof_with_goal.splitlines()[1:])
    proofs_extracted_text = give_all_extraction_text(proof)
    proofs_extracted = [
        p.strip()
        for p in re.split(r'\n\s*\n', proofs_extracted_text.strip())
        if p.strip()
    ]

    # walidacja
    for p in proofs_extracted:
        check_proof_with_assumptions(p)

    return proofs_extracted


poofs = []

with ProcessPoolExecutor() as ex:
    results = list(tqdm(ex.map(_extract_and_check_one, text), total=len(text)))

poofs = list(chain.from_iterable(results))


In [ ]:
def replace_one_idx(proof, line_number, number_in_line, new_number):#numery linii numeruję od 1, numery w linii od 0
    def line_idx(line):
        return int(re.match(r"\d+", s).group())
    proof = proof.splitlines()
    def replace_nth_number(text: str, n: int, x: int, pattern: str = r"(–|, )(\d+)") -> str:
        counter = {"i": 0}

        def repl(match):
            i = counter["i"]
            counter["i"] += 1
            if i == n:
                return match.group(1) + str(x)
            return match.group(0)
        return re.sub(pattern, repl, text)
    for i in range(len(proof)):
        if proof[i].startswith(str(line_number) + ". "):
            proof[i] = replace_nth_number(proof[i], number_in_line, new_number)
    return "\n".join(proof)


def replace_one_number_by_smallest_number(proof,line_number,number_in_line):
    max_iter = len(proof.splitlines())
    for i in range(0,max_iter+1):
        proof = replace_one_idx(proof,line_number,number_in_line,i)
        try:
            check_proof_with_assumptions(proof)
            return proof
        except:
            pass


def replace_all_number_by_smallest_number(proof):
    number_of_numbers = []
    proof_sole = ""
    proof_splited = proof.splitlines()
    for i in range(len(proof_splited)):
        if proof_splited[i][0] in ["1","2","3","4","5","6","7","8","9"]:
            proof_sole += proof_splited[i]
            proof_sole += "\n"
    proof_sole = proof_sole[:-1]

    proof_splited = proof_sole.splitlines()
    for i in proof_splited:
        number_of_numbers.append(len([x for x in re.findall(r"(–|, )(\d+)", i)]))
    for i in range(len(proof_splited),0,-1):
        for j in range(number_of_numbers[i-1]):
            proof = replace_one_number_by_smallest_number(proof,i,j)
    return proof

with ProcessPoolExecutor(max_workers=os.cpu_count()) as executor:
    proofs_with_optimized_numbers = list(
        tqdm(
            executor.map(
                replace_all_number_by_smallest_number,
                proofs, #text
                chunksize=320
            ),
            total=len(proofs)
        )
    )

In [ ]:
def is_taken_from_context(proof):
    proofParsed = check_proof_with_assumptions(proof)
    base_context = proofParsed[1].LP.assumptions.base_context
    goal = proofParsed[len(proofParsed)].LP.conclusion
    if  goal in base_context:
        return True
    else:
        return False

proofs_without_trivial = []

for i in tqdm(range(len(proofs_with_optimized_numbers))):
    if not is_taken_from_context(proofs_with_optimized_numbers[i]):
        proofs_without_trivial.append(proofs_with_optimized_numbers[i])



from concurrent.futures import ProcessPoolExecutor
from tqdm.auto import tqdm


def is_taken_from_context(proof):
    proofParsed = check_proof_with_assumptions(proof)
    base_context = proofParsed[1].LP.assumptions.base_context
    goal = proofParsed[len(proofParsed)].LP.conclusion
    return goal in base_context


with ProcessPoolExecutor(max_workers=15) as ex:
    mask = list(tqdm(
        ex.map(is_taken_from_context, proofs_with_optimized_numbers, chunksize=50),
        total=len(proofs_with_optimized_numbers)
    ))

proofs_without_trivial = [
    proof
    for proof, taken in zip(proofs_with_optimized_numbers, mask)
    if not taken
]

In [ ]:
def remove_extra_lines_at_the_end(proof):
    parsedProof = check_proof_with_assumptions(proof)
    smallest_last_idx = len(parsedProof)

    for i in parsedProof.keys():
        if parsedProof[i].LP == parsedProof[smallest_last_idx].LP:
            if i < smallest_last_idx:
                smallest_last_idx = i

    ans = ""
    splitted_proof = proof.splitlines()
    current_idx = 0

    for i in range(len(splitted_proof)):
        ans += splitted_proof[i]

        if current_idx >= 1:
            current_idx += 1

        if splitted_proof[i][0] == "1" and current_idx == 0:
            current_idx += 1

        if current_idx == smallest_last_idx:
            check_proof_with_assumptions(ans)
            return ans

        ans += "\n"


with ProcessPoolExecutor(max_workers=30) as ex:
    proofs_without_extra_lines_at_the_end = list(
        tqdm(
            ex.map(remove_extra_lines_at_the_end, proofs_without_trivial, chunksize=50),
            total=len(proofs_without_trivial)
        )
    )

In [ ]:
def extract_proof_with_assumptions(proof):
    proof_splitted = proof.splitlines()
    ans = ""
    for i in range(len(proof_splitted)):
        if proof_splitted[i][0] == '1':
            break
        ans += proof_splitted[i]
        ans += "\n"
    neighbours = dict()
    line_text = dict()
    for i in range(len(proof_splitted)):
        if proof_splitted[i][0] in ['1','2','3','4','5','6','7','8','9']:
            idx = int(re.findall(r"(\d+)(\.)",proof_splitted[i])[0][0])
            line_text[idx] = proof_splitted[i]
            neighbours[idx] = [int(x[1]) for  x  in re.findall(r"(–|, )(\d+)",proof_splitted[i])]
            neighbours[idx] = [x for x in neighbours[idx] if x != 0]
    idxs = {len(neighbours)}
    new_idxs = {len(neighbours)}
    while new_idxs != set():
        i = random.choice(list(new_idxs))
        new_idxs.remove(i)
        for j in neighbours[i]:
            if not j in idxs:
                idxs.add(j)
                new_idxs.add(j)
    for i in range(len(proof_splitted)):
        if proof_splitted[i][0] in ['1','2','3','4','5','6','7','8','9']:
            idx = int(re.findall(r"(\d+)(\.)",proof_splitted[i])[0][0])
            if idx in idxs:
                ans += line_text[idx]
                ans += "\n"
    ans = ans[:-1]
    old_numbers = sorted(list(idxs))
    new_numbers = list(range(1,len(old_numbers)+1))
    def replace_selected_numbers(s, old_numbers, new_numbers):
        if len(old_numbers) != len(new_numbers):
            raise ValueError("old_numbers i new_numbers muszą mieć tę samą długość")
        mapping = {str(old): str(new) for old, new in zip(old_numbers, new_numbers)}
        pattern = r'(?:(?<=–)\d+|(?<=, )\d+|\d+(?=\.))'
        def repl(match):
            num = match.group(0)
            return mapping.get(num, num)
        return re.sub(pattern, repl, s)
    ans = replace_selected_numbers(ans, old_numbers, new_numbers)
    check_proof_with_assumptions(ans)
    return ans

with ProcessPoolExecutor(max_workers=30) as ex:
    proofs_extracted = list(
        tqdm(
            ex.map(
                extract_proof_with_assumptions,
                proofs_without_extra_lines_at_the_end,
                chunksize=50
            ),
            total=len(proofs_without_extra_lines_at_the_end)
        )
    )

In [ ]:
TEXT = "\n\n".join(proofs_extracted)

with open("NEW_CORPUS_PROVING.txt", "w", encoding="utf-8") as f:
    f.write(TEXT)